# Code 10.4.1: Monte Carlo prediction

Generates episodes of the uniform random policy from random non-terminal start cells, walks each episode backward to compute returns, and moves each visited state's estimate toward its return. With `alpha=None` the step size is $`1/N(s)`$, giving the running mean.


In [ ]:
import sys
from pathlib import Path

def _add_gridworld_path():
    """The chapter figure scripts ship the gridworld module this listing imports."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / "chapters/10-reinforcement-learning-basics/figures/src/gridworld.py"
        if candidate.is_file():
            folder = str(candidate.parent)
            if folder not in sys.path:
                sys.path.insert(0, folder)
            return
    raise FileNotFoundError(
        "Could not find chapters/10-reinforcement-learning-basics/figures/src/gridworld.py. "
        "Run this notebook from the repository (or a subdirectory of it)."
    )

_add_gridworld_path()


In [ ]:
import numpy as np
from gridworld import GridWorld

env = GridWorld()
GAMMA = 0.9
nonterminal = np.flatnonzero(~env.terminal)

def generate_episode(env, rng):
    """One episode of the uniform random policy; returns states and rewards."""
    s = rng.choice(nonterminal)
    states, rewards = [], []
    done = False
    while not done:
        a = rng.integers(env.n_actions)
        s2, r, done = env.step(s, a)
        states.append(s)
        rewards.append(r)
        s = s2
    return states, rewards

def mc_prediction(env, n_episodes, rng, alpha=None, gamma=GAMMA):
    """Every-visit Monte Carlo: move V(s) toward each observed return G_t."""
    V, N = np.zeros(env.n_states), np.zeros(env.n_states)
    history = []
    for _ in range(n_episodes):
        states, rewards = generate_episode(env, rng)
        G = 0.0
        for s, r in zip(reversed(states), reversed(rewards)):   # walk backward
            G = r + gamma * G                                   # return from s
            N[s] += 1
            step = 1.0 / N[s] if alpha is None else alpha
            V[s] += step * (G - V[s])
        history.append(V.copy())
    return np.array(history)
